# Model Training — Fussball Vorhersagen

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle
print('Libraries geladen!')

## 1. Features laden

In [ ]:
df = pd.read_csv('data/features.csv')
print(f'Spiele geladen: {len(df)}')
df.head()

## 2. Daten vorbereiten

In [ ]:
FEATURES = ['home_form', 'away_form', 'form_diff', 'h2h', 'heimquote']
TARGET   = 'winner'

df = df.dropna(subset=FEATURES + [TARGET])
print(f'Spiele nach Bereinigung: {len(df)}')

label_map = {'HOME_TEAM': 1, 'DRAW': 0, 'AWAY_TEAM': -1}
df['target'] = df[TARGET].map(label_map)

print('Ergebnis-Verteilung:')
print(df['winner'].value_counts())

## 3. Train/Test Split — zeitlich

In [ ]:
split = int(len(df) * 0.8)

train = df.iloc[:split]
test  = df.iloc[split:]

X_train = train[FEATURES]
y_train = train['target']
X_test  = test[FEATURES]
y_test  = test['target']

print(f'Training:  {len(train)} Spiele')
print(f'Test:      {len(test)} Spiele')

## 4. Baseline — immer Heimsieg tippen

In [ ]:
baseline_pred = [1] * len(y_test)
baseline_acc  = accuracy_score(y_test, baseline_pred)
print(f'Naive Baseline (immer Heimsieg): {baseline_acc*100:.1f}%')

## 5. Logistische Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_acc  = accuracy_score(y_test, lr_pred)

print(f'Logistische Regression: {lr_acc*100:.1f}%')
print()
print(classification_report(y_test, lr_pred, target_names=['Auswärtssieg', 'Unentschieden', 'Heimsieg']))

## 6. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_acc  = accuracy_score(y_test, rf_pred)

print(f'Random Forest: {rf_acc*100:.1f}%')
print()
print(classification_report(y_test, rf_pred, target_names=['Auswärtssieg', 'Unentschieden', 'Heimsieg']))

## 7. Vergleich

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

modelle = ['Baseline', 'Log. Regression', 'Random Forest']
werte   = [baseline_acc, lr_acc, rf_acc]
farben  = ['#cccccc', '#2a78d6', '#1baf7a']

axes[0].bar(modelle, [v*100 for v in werte], color=farben)
axes[0].set_title('Genauigkeit der Modelle (%)')
axes[0].set_ylim(0, 100)
for i, v in enumerate(werte):
    axes[0].text(i, v*100 + 1, f'{v*100:.1f}%', ha='center')

importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[1], color='#1baf7a')
axes[1].set_title('Feature Wichtigkeit (Random Forest)')

plt.tight_layout()
plt.show()

print(f'Bestes Modell: {modelle[werte.index(max(werte))]} mit {max(werte)*100:.1f}%')

## 8. Confusion Matrix

In [ ]:
best_pred = rf_pred if rf_acc > lr_acc else lr_pred
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Auswärtssieg', 'Unentschieden', 'Heimsieg'],
            yticklabels=['Auswärtssieg', 'Unentschieden', 'Heimsieg'])
plt.title('Confusion Matrix')
plt.ylabel('Echtes Ergebnis')
plt.xlabel('Vorhersage')
plt.tight_layout()
plt.show()

## 9. Modell speichern

In [ ]:
best_model = rf if rf_acc > lr_acc else lr
model_name = 'Random Forest' if rf_acc > lr_acc else 'Logistische Regression'

with open('data/model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print(f'Modell gespeichert: {model_name}')
print(f'Genauigkeit: {max(rf_acc, lr_acc)*100:.1f}%')
print('Datei: data/model.pkl')

## 10. Manuelle Vorhersage testen

In [ ]:
beispiel = pd.DataFrame([{
    'home_form':  12,
    'away_form':  6,
    'form_diff':  6,
    'h2h':        0.6,
    'heimquote':  0.7
}])

vorhersage = best_model.predict(beispiel)[0]
wahrscheinlichkeiten = best_model.predict_proba(beispiel)[0]

label_map_reverse = {1: 'Heimsieg', 0: 'Unentschieden', -1: 'Auswärtssieg'}
klassen = best_model.classes_

print(f'Vorhersage: {label_map_reverse[vorhersage]}')
print()
print('Wahrscheinlichkeiten:')
for klasse, prob in zip(klassen, wahrscheinlichkeiten):
    print(f'  {label_map_reverse[klasse]}: {prob*100:.1f}%')